In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

**<font size="6" color="red">ch1_허깅페이스 모델 사용하기</font>**
- Inference API 이용 : 모델의 결과를 server에서
- pipeline() 이용 : 모델을 다운로드 받아 모델의 결과를 local에서  

- 허깅페이스 transformer에서 지원하는 task

| task값| 설명 |  
| : ---| : ---  |
|text-classification(별칭 sentimentanalysis)    | 감정 분석, 뉴스 분류, 리뷰 분류 등 문장 분류|
|zero-shot-classification                      |레이블에 대한 별도 학습 없이 후보 레이블 중에서 분류|
|text-generation                               |GPT 계열 모델을 이용한 텍스트 생성|
|fill-mask                                     |문장 안의 빈칸(마스크)에 들어갈 단어 예측|
|ner (token-classification의 별칭)              |개체명 인식(사람, 조직, 장소 등 라벨링)|
|question-answering                            |주어진 지문(context)을 근거로 질문에 답변|
|summarization                                 |긴 문서를 짧게 요약|
|translation                                   |서로 다른 언어 간 번역|
|image-to-text                                 |이미지 내용을 설명하는 문장 생성|
|image-classification                          |이미지가 어떤 대상인지 분류|

- 처음 모델 사용시 "c:/Users/내컴퓨터이름/.cache/huggingface"에 다운로드되는라 시간이 걸림


In [4]:
import warnings 
import os 
import logging 
# 경고 메시지 제거 warnings.filterwarnings('ignore') 
# transformers 라이브러리의 로깅 레벨을 ERROR로 조정 (경고 숨김) logging.getLogger("transformers").setLevel(logging.ERROR) 
# Hugging Face 캐시 관련 symlink 경고 제거 os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1' os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

## 1. 텍스트 기반 감정분석(긍정/부정)
- 토큰화 -> 워드임베딩 -> 모델 -> predict : pipeline()함수는 이단계를 내부적으로 해 줌


In [3]:
from transformers import pipeline
classifier = pipeline(task="text-classification",
                     model ="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
classifier("I've been waiting for a Hugging face course my whole life")

C:\Users\mbc\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982088804244995}]

In [2]:
from transformers import AutoModel

model = AutoModel.from_pretrained("distilbert/distilbert-base-uncased-finetuned-sst-2-english")

# 전체 파라미터 개수 세기
total_params = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터 수: {total_params:,}")
print(f"전체 파라미터 수: {total_params/1024/1024:.3f}MB")

C:\Users\mbc\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


전체 파라미터 수: 66,362,880
전체 파라미터 수: 63.289MB


In [6]:
classifier = pipeline(task="sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
#감정분석할 내용이 많으면 list
classifier([
    "I've been waiting for a Hugging face course my whole life",
    "I hate this so much!"
])

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9982088804244995},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

In [9]:
classifier([
    "이 영화는 그저그러네",
    "이 영화는 그렇게 재밌지 않았지만 캐스팅이 더 별로, 근데 시간 많은사람만 봐"
])

[{'label': 'POSITIVE', 'score': 0.896101713180542},
 {'label': 'POSITIVE', 'score': 0.7968655824661255}]

In [10]:
classifier([
    "I like you",
    "힘들어요"
])

[{'label': 'POSITIVE', 'score': 0.9998695850372314},
 {'label': 'POSITIVE', 'score': 0.8669533729553223}]

In [11]:
classifier = pipeline(task="sentiment-analysis",
                      model="daekeun-ml/koelectra-small-v3-nsmc")
texts=['힘듭니다','오늘 기분 좋아', '당신이 싫지 않습니다']
classifier(texts)

C:\Users\mbc\anaconda3\envs\llm\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mbc\.cache\huggingface\hub\models--daekeun-ml--koelectra-small-v3-nsmc. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu


[{'label': '0', 'score': 0.99137282371521},
 {'label': '1', 'score': 0.9968572854995728},
 {'label': '1', 'score': 0.975990355014801}]

In [16]:
for text, result in zip(texts, classifier(texts)):
   label ="긍정" if result['label']=='1' else "부정"
   print(f"'{text}' -> {label} {result['score']:.2%}")
    

'힘듭니다' -> 부정 99.14%
'오늘 기분 좋아' -> 긍정 99.69%
'당신이 싫지 않습니다' -> 긍정 97.60%


## 2. 제로샷(Zero-shot-분류)
- 비지도학습
```
제로샷 분류는 기계학습 및 자연어 처리에서 개별 작업에 대한 별도의 학습(파인튜닝) 없이도 분류 작업을 수행할 수 있는 방식이다. 분류하고자 하는 후보 레이블(candidate_labels)만 지정해 주면, 모델이 사전에 학습한 언어 지식을 바탕으로 입력 문장이 어떤 레이블에 가장 가까운지 확률로 계산해 준다.
```


In [ ]:
classifier =pipeline(task='zero-shot-classification',
                    model='facebook/bart-large-mnli')
classifier("I have a problem with my iphone that needs to be resolved asap!!",
          candidate_labels=['phone', 'urgent', 'tablet', 'computer'])


In [ ]:
classifier("This is a course about the Transformer library.",
          candidate_labels=['education', 'business', 'phone'])